# 1. Setup
**Baseline 1: Independent CNNs.** Two separate CNNs. One predicts the crop, the other predicts the disease.
A pair score is log P(crop) + log P(disease), so unseen pairs get a score too.

In [1]:
import copy
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))  # lets the notebook import common/

import timm
import torch
import torch.nn as nn
import torch.nn.functional as F

from common.config import CHECKPOINT_DIR, DEVICE, NUM_CROPS, NUM_DISEASES, set_seed
from common.dataset import get_loaders, get_pairs
from common.evaluate import evaluate, val_accuracy

MODEL_NAME = "baseline1_independent"
DEBUG = True  # True: small data, 1 epoch. Set False for the full run.
HP = {"backbone": "resnet50", "lr": 1e-4, "weight_decay": 1e-4, "epochs": 20,
      "batch_size": 64, "optimizer": "AdamW", "label_smoothing": 0.1}
EPOCHS = 1 if DEBUG else HP["epochs"]
set_seed(42)

# 2. Data

In [ ]:
train_loader, val_loader, seen_loader, unseen_loader = get_loaders(
    HP["batch_size"], norm="imagenet", debug=DEBUG)
pairs = get_pairs().to(DEVICE)

# 3. Model

In [ ]:
def make_classifier(num_classes):
    return timm.create_model(HP["backbone"], pretrained=True, num_classes=num_classes)


crop_model = make_classifier(NUM_CROPS).to(DEVICE)
disease_model = make_classifier(NUM_DISEASES).to(DEVICE)


@torch.no_grad()
def predict(images):
    crop_model.eval()
    disease_model.eval()
    images = images.to(DEVICE)
    crop_log_prob = F.log_softmax(crop_model(images).float(), dim=1)
    disease_log_prob = F.log_softmax(disease_model(images).float(), dim=1)
    # Treat crop and disease as independent, so pair score = sum of log-probs
    return crop_log_prob[:, pairs[:, 0]] + disease_log_prob[:, pairs[:, 1]]

# 4. Training

In [ ]:
# The two models share no weights and no loss, so one optimizer still trains them independently
parameters = list(crop_model.parameters()) + list(disease_model.parameters())
optimizer = torch.optim.AdamW(parameters, lr=HP["lr"], weight_decay=HP["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss(label_smoothing=HP["label_smoothing"])
best_val, best_state = -1.0, None

for epoch in range(EPOCHS):
    crop_model.train()
    disease_model.train()
    for images, crop_labels, disease_labels, _ in train_loader:
        images = images.to(DEVICE)
        crop_labels, disease_labels = crop_labels.to(DEVICE), disease_labels.to(DEVICE)
        with torch.autocast(DEVICE.type, dtype=torch.bfloat16, enabled=DEVICE.type == "cuda"):
            crop_loss = criterion(crop_model(images), crop_labels)
            disease_loss = criterion(disease_model(images), disease_labels)
        optimizer.zero_grad()
        (crop_loss + disease_loss).backward()
        optimizer.step()
    scheduler.step()
    val_top1 = val_accuracy(predict, val_loader)
    print(f"epoch {epoch + 1}: crop loss {crop_loss.item():.3f}, "
          f"disease loss {disease_loss.item():.3f}, val top-1 {val_top1:.2f}%")
    if val_top1 > best_val:  # keep the best epoch by val, never by test
        best_val = val_top1
        best_state = copy.deepcopy({"crop": crop_model.state_dict(), "disease": disease_model.state_dict()})

crop_model.load_state_dict(best_state["crop"])
disease_model.load_state_dict(best_state["disease"])

# 5. Evaluation

In [ ]:
results = evaluate(predict, seen_loader, unseen_loader, MODEL_NAME, HP, robust=True)

# 6. Save

In [ ]:
CHECKPOINT_DIR.mkdir(exist_ok=True)
torch.save({"model_name": MODEL_NAME, "state_dict": best_state, "hp": HP},
           CHECKPOINT_DIR / f"{MODEL_NAME}.pt")